# Assignment 5 — Option B: Job Fit Analyzer
## BSAN 6200: Text Mining & Social Media Analytics — Spring 2026

**Student Name:** Shahzeb Ather  
**Date:** 04/24/2026
**Option:** B — Job Fit Analyzer  
**API Path:** Free

---

### Table of Contents
1. [Setup and Imports](#1-setup)
2. [Load Job Descriptions and Resume](#2-loading)
3. [Text Chunking](#3-chunking)
4. [Embedding and Vector Store](#4-embedding)
5. [Analysis Prompts and Chain](#5-analysis)
6. [Zero-shot vs. Few-shot Comparison](#6-comparison)
7. [Evaluation](#7-evaluation)

> **Reminder:** The Streamlit app is a separate file (`streamlit_app.py`). This notebook builds and tests the analysis pipeline.  
> See the Option B Implementation Guide for detailed step requirements.

---
<a id="1-setup"></a>
## 1. Setup and Imports

Install required packages and load your API key from a `.env` file.  
**Do NOT hardcode API keys in this notebook.**

Suggested packages: `langchain`, `langchain-openai` or `langchain-community`, `chromadb` or `faiss-cpu`, `pypdf`, `python-dotenv`, `pandas`, `sentence-transformers` (free path)

In [ ]:
#Install packages
!python3 -m pip install streamlit chromadb huggingface-hub python-dotenv sentence-transformers pypdf pandas jupyterlab ipykernel

In [2]:
#Load API keys from .env
import os
import pandas as pd
from dotenv import load_dotenv
from pypdf import PdfReader
import chromadb
from huggingface_hub import InferenceClient

load_dotenv()

# ── Your imports below ──


True

---
<a id="2-loading"></a>
## 2. Load Job Descriptions and Resume

**Required:**
- 10+ JD files in `data/job_descriptions/` (each as a separate .txt or .pdf)
- Your resume in `data/resume/`
- A metadata file `data/jd_metadata.csv` with columns: filename, company, title, source_url, date_collected

Print: number of JDs loaded, number of resume docs, and preview content from each.

In [ ]:
# ── Load JD metadata ──
from pathlib import Path
import os

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

meta_path = ROOT / "data" / "jd_metadata.csv"
if meta_path.exists():
    jd_meta = pd.read_csv(meta_path)
    display(jd_meta)
else:
    jd_meta = pd.DataFrame()
    print("No data/jd_metadata.csv — app will fall back to filenames only.")


In [ ]:
# ── Load JD documents and resume ──
# (Same helpers as streamlit_app.py — notebook cannot import that file without launching Streamlit UI.)

def load_text_file(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return f.read()

def load_pdf_file(filepath):
    from pypdf import PdfReader
    reader = PdfReader(filepath)
    return "\n".join(page.extract_text() or "" for page in reader.pages)

def load_file(filepath):
    return load_pdf_file(filepath) if str(filepath).endswith(".pdf") else load_text_file(filepath)

def load_all_jds(jd_dir="data/job_descriptions"):
    docs = []
    p = ROOT / jd_dir
    if not p.exists():
        return docs
    for filename in sorted(os.listdir(p)):
        fp = p / filename
        if filename.endswith((".txt", ".pdf")):
            text = load_file(fp)
            if text.strip():
                docs.append({"text": text, "source": filename})
    return docs

def load_resume(resume_dir="data/resume"):
    p = ROOT / resume_dir
    if not p.exists():
        return ""
    for filename in os.listdir(p):
        if filename.endswith((".txt", ".pdf")):
            return load_file(p / filename)
    return ""

jd_docs = load_all_jds()
resume_text = load_resume()
print(f"Loaded {len(jd_docs)} JD files, resume chars: {len(resume_text)}")


In [ ]:
# ── Preview sample content ──
if jd_docs:
    d0 = jd_docs[0]
    print(d0["source"], "chars:", len(d0["text"]))
    print(d0["text"][:800], "\n…")
else:
    print("No JDs in data/job_descriptions/")
print("\nResume preview:\n", (resume_text[:600] + "…") if len(resume_text) > 600 else resume_text)


---
<a id="3-chunking"></a>
## 3. Text Chunking

Split your documents into chunks.  
**Required:** Try at least 2 chunking strategies, compare them quantitatively, and justify your final choice.

**Hint:** JDs often have natural sections (Requirements, Responsibilities, Qualifications). Consider whether your splitter respects these boundaries.

In [ ]:
# ── Strategy 1: paragraph merge + overlap (default in app) ──

def chunk_documents_paragraph(documents, max_chars=1200, overlap=150):
    chunks = []
    for doc in documents:
        text = doc["text"].strip()
        source = doc["source"]
        paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
        current = ""
        for p in paragraphs:
            if len(p) > max_chars:
                for part in p.replace(". ", ".\n").split("\n"):
                    part = part.strip()
                    if not part:
                        continue
                    if len(current) + len(part) + 2 <= max_chars:
                        current = f"{current}\n\n{part}".strip()
                    else:
                        if current:
                            chunks.append({"text": current, "source": source})
                        tail = current[-overlap:] if overlap and current else ""
                        current = f"{tail}\n\n{part}".strip() if tail else part
            else:
                if len(current) + len(p) + 2 <= max_chars:
                    current = f"{current}\n\n{p}".strip() if current else p
                else:
                    if current:
                        chunks.append({"text": current, "source": source})
                    tail = current[-overlap:] if overlap and current else ""
                    current = f"{tail}\n\n{p}".strip() if tail else p
        if current:
            chunks.append({"text": current, "source": source})
    return chunks

para_chunks = chunk_documents_paragraph(jd_docs)
print("Paragraph strategy:", len(para_chunks), "chunks")


In [ ]:
# ── Strategy 2: fixed character windows ──

def chunk_documents_fixed_window(documents, window=900, stride=600):
    chunks = []
    for doc in documents:
        text = doc["text"].strip()
        source = doc["source"]
        if not text:
            continue
        start = 0
        while start < len(text):
            piece = text[start : start + window]
            if piece.strip():
                chunks.append({"text": piece.strip(), "source": source})
            start += stride
    return chunks

win_chunks = chunk_documents_fixed_window(jd_docs)
print("Fixed-window strategy:", len(win_chunks), "chunks")


In [ ]:
# ── Compare strategies (quick quantitative view) ──
import statistics as stats

def stats_for(chunks):
    lens = [len(c["text"]) for c in chunks]
    return {
        "n_chunks": len(lens),
        "mean_len": round(stats.mean(lens), 1) if lens else 0,
        "median_len": int(stats.median(lens)) if lens else 0,
    }

print("Paragraph:", stats_for(para_chunks))
print("Fixed window:", stats_for(win_chunks))


### Chunking Decision

**Which strategy did you choose?**  
**Why?**  
**Final settings (chunk_size, overlap):**

---
<a id="4-embedding"></a>
## 4. Embedding and Vector Store

Embed your chunks and store them in a vector database (ChromaDB or FAISS).

**Paid path:** OpenAI `text-embedding-3-small`  
**Free path:** `sentence-transformers/all-MiniLM-L6-v2`

After creating the store, run a test similarity search to verify it works.

In [ ]:
# Embeddings: Chroma default embedding function (Sentence Transformers) is used when you call collection.add(documents=...).
# No separate OpenAI call in this assignment path — see streamlit_app.py load_vectorstore() for the live app.
import chromadb


In [ ]:
# ── Create embeddings and vector store ──
TOP_K_CHUNKS = 5
chunks = chunk_documents_paragraph(jd_docs)  # default choice matches Streamlit app

client = chromadb.Client()
try:
    client.delete_collection("job_fit_nb")
except Exception:
    pass
collection = client.create_collection("job_fit_nb")
collection.add(
    documents=[c["text"] for c in chunks],
    metadatas=[{"source": c["source"]} for c in chunks],
    ids=[f"chunk_{i}" for i in range(len(chunks))],
)
print("Indexed", len(chunks), "chunks into Chroma collection job_fit_nb")


In [ ]:
# ── Verify: run a test similarity search ──
if jd_docs:
    fname = jd_docs[0]["source"]
    q = "skills requirements qualifications " + resume_text[:400]
    res = collection.query(query_texts=[q], n_results=5, where={"source": fname})
    for i, doc in enumerate(res["documents"][0]):
        print(f"--- hit {i+1} ---\n", doc[:500], "\n")


---
<a id="5-analysis"></a>
## 5. Analysis Prompts and Chain

Build 3 analysis types, each with its own prompt:

1. **Skill Gap Report:** Compare resume skills vs. JD requirements. Output matching skills, missing skills, and recommended actions.
2. **Keyword Alignment:** Extract key terms from a JD, check which appear in the resume, report a match rate.
3. **Fit Summary:** 3-4 sentence narrative assessment citing evidence from both documents.

You also need to wire up the LLM and a way to pass a specific JD + resume into each prompt.

**Required:** Document at least 3 prompt iterations total (across any analysis type) with rationale.

**Reminder:** Prompt design must be your own work (Tier 2 — AI prohibited for this step).

In [ ]:
# ── Initialize LLM ──
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
token = os.environ.get("HF_TOKEN", "")
if not token:
    print("Set HF_TOKEN in .env at project root to run live generations from the notebook.")
    hf_client = None
else:
    hf_client = InferenceClient(token=token)
    print("InferenceClient ready for", MODEL_ID)


In [ ]:
# ── Analysis 1: Skill Gap Report (prompt v3) ──
SKILL_GAP_PROMPT = """
You are a job-fit evaluator.

Task:
Compare the candidate resume to the job description and produce a skill gap report.

Rules:
- Use only the provided resume and JD text.
- Do not invent experience or certifications.
- Be concise and specific.
- For every strength and every gap, include one short quote from the JD or resume as evidence.
- Return exactly 3 strengths and exactly 3 gaps.
- For each gap, start the bullet with a severity label in this exact form: **[Severity: High]**, **[Severity: Medium]**, or **[Severity: Low]**.
  Use **High** when the JD states must-have / required / minimum years / core responsibility language; **Medium** for important but not clearly mandatory; **Low** for nice-to-have or bonus language.

Output:
1) Matching Skills (exactly 3 bullets)
2) Missing Skills / Gaps (exactly 3 bullets; each bullet begins with **[Severity: High|Medium|Low]** then the gap text and evidence quote)
3) Recommended Actions (exactly 3 bullets; each maps to one gap or one theme, no filler)
4) Fit Score (0-100) with a short explanation
"""
print("Skill Gap prompt length:", len(SKILL_GAP_PROMPT))


In [ ]:
# ── Analysis 2: Keyword Alignment ──
KEYWORD_PROMPT = """
You are a keyword alignment assistant.

Task:
Extract important keywords from the job description and check whether they appear
(or close equivalent appears) in the candidate resume.

Rules:
- Use only provided text.
- Avoid over-claiming matches.

Output:
1) Top JD Keywords (10-20)
2) Matched Keywords
3) Missing Keywords
4) Keyword Match Rate (%) = matched / total * 100
5) Top 3 keywords to add or emphasize in resume
"""
print("Keyword prompt length:", len(KEYWORD_PROMPT))


In [ ]:
# ── Analysis 3: Fit Summary ──
FIT_SUMMARY_PROMPT = """
You are a hiring-side reviewer.

Task:
Write a brief fit summary comparing this resume to the job description.

Rules:
- Use evidence from both texts.
- Keep it realistic and balanced.
- No hallucinations.

Output:
- Fit Summary (3-4 sentences)
- Top 3 Strengths (bullets)
- Top 3 Concerns (bullets)
- Recommendation: Strong Fit / Moderate Fit / Low Fit
"""
print("Fit Summary prompt length:", len(FIT_SUMMARY_PROMPT))


### Prompt Iteration Log

Document at least 3 total iterations across any of the analysis types.

**Iteration 1:** [Which analysis? What changed? Why? What improved?]

**Iteration 2:** [Which analysis? What changed? Why? What improved?]

**Iteration 3:** [Which analysis? What changed? Why? What improved?]

---
<a id="6-comparison"></a>
## 6. Zero-shot vs. Few-shot Comparison

Pick one of your 3 analysis types. Create a few-shot version by adding 1-2 example input/output pairs to the prompt. Run both versions on the same JD and compare outputs.

**Reminder:** You must write the few-shot examples yourself (Tier 2).

In [ ]:
# ── Few-shot: add 1 compact example to Skill Gap (optional pattern) ──
# Uncomment to prepend a tiny few-shot block before your real prompt in the app.
FEW_SHOT_PREFIX = """
Example format (illustrative):
Strength: ... Evidence: \"...\"
"""
print(len(FEW_SHOT_PREFIX))


In [ ]:
# ── Zero-shot vs few-shot: print side-by-side prompt heads (no API spend) ──
print("ZERO_SHOT head:\n", SKILL_GAP_PROMPT[:400])
print("\n---\nFEW_SHOT head:\n", (FEW_SHOT_PREFIX + SKILL_GAP_PROMPT)[:500])


### Zero-shot vs. Few-shot Analysis

**Which analysis type did you compare?**

**Which performed better?**

**Why? (use specific examples from the outputs above)**

---
<a id="7-evaluation"></a>
## 7. Evaluation

Run all 3 analysis types on your **top 3 target JDs** (9 total analyses).

For each, score:
- **Retrieval relevance:** Did it pull the right JD sections? (Yes/Partial/No)
- **Skill identification accuracy:** Are identified skills/gaps correct? (count correct vs. incorrect)
- **Actionability:** Are recommendations specific and useful? (1-5)
- **Faithfulness:** Does output stick to document content? (Faithful/Partial/Hallucinated)

**Reminder:** Evaluation must be your own work (Tier 2 — AI prohibited).

In [ ]:
# ── Batch idea: 3 JDs × 3 analysis types (expensive on HF — scaffold only) ──
ANALYSIS_TYPES = {
    "Skill Gap Report": SKILL_GAP_PROMPT,
    "Keyword Alignment": KEYWORD_PROMPT,
    "Fit Summary": FIT_SUMMARY_PROMPT,
}
subset = jd_docs[:3]
print("Would run", len(subset) * len(ANALYSIS_TYPES), "calls for first", len(subset), "postings — execute in app or add a loop + sleep when token budget allows.")


In [ ]:
# ── Summarize evaluation results (point to written log) ──
from pathlib import Path
p = ROOT / "evaluation" / "test_results.md"
print(p.exists(), p)
if p.exists():
    txt = p.read_text(encoding="utf-8")
    print("test_results.md chars:", len(txt))


### Evaluation Analysis

**Which analysis type worked best?**

**Which JDs produced the best/worst results? Why?**

**Where did the system hallucinate or produce inaccurate results?**

**What would you improve?**

---

## Next Steps

1. Build your Streamlit app (`streamlit_app.py`) using the pipeline from this notebook
2. Write your Technical Manager Memo (`memo.md`)
3. Complete your AI Usage Log (`ai_log.md`)
4. Verify GitHub repository structure and commit count

---
*BSAN 6200 | Spring 2026 | Assignment 5 — Option B*

## Sample Prompt Iteration (No Real JD Files Yet)
Use this section to practice prompt design with 1-2 sample job texts before loading the full JD set.

In [20]:
sample_resume = '''
Data analyst with 2 years of experience in Python, SQL, Tableau, and dashboard development.
Built KPI reporting pipelines and worked with cross-functional business stakeholders.
'''

sample_jd_1 = '''
Business Analyst role: requires SQL, Excel, dashboarding, stakeholder communication, and process documentation.
Preferred: Python and A/B testing exposure.
'''

sample_jd_2 = '''
Data Analyst role: requires Python, SQL, ETL workflow knowledge, Tableau/Power BI, and experimentation support.
Preferred: cloud data warehouse experience.
'''

print('Sample inputs loaded')

Sample inputs loaded


In [21]:
prompt_v1 = f'''
Compare the resume and job description.
Give a fit score out of 100, 3 strengths, and 3 gaps.

Resume:
{sample_resume}

Job Description:
{sample_jd_1}
'''

prompt_v2 = f'''
You are a job-fit evaluator. Use only evidence from the inputs.
Output JSON keys: fit_score, strengths, gaps, improvement_actions.
Each strength and gap must include a short evidence quote.
Do not invent experience not present in the resume.

Resume:
{sample_resume}

Job Description:
{sample_jd_1}
'''

print('Prompt v1:\n', prompt_v1[:400], '...')
print('\nPrompt v2:\n', prompt_v2[:500], '...')

Prompt v1:
 
Compare the resume and job description.
Give a fit score out of 100, 3 strengths, and 3 gaps.

Resume:

Data analyst with 2 years of experience in Python, SQL, Tableau, and dashboard development.
Built KPI reporting pipelines and worked with cross-functional business stakeholders.


Job Description:

Business Analyst role: requires SQL, Excel, dashboarding, stakeholder communication, and process  ...

Prompt v2:
 
You are a job-fit evaluator. Use only evidence from the inputs.
Output JSON keys: fit_score, strengths, gaps, improvement_actions.
Each strength and gap must include a short evidence quote.
Do not invent experience not present in the resume.

Resume:

Data analyst with 2 years of experience in Python, SQL, Tableau, and dashboard development.
Built KPI reporting pipelines and worked with cross-functional business stakeholders.


Job Description:

Business Analyst role: requires SQL, Excel, dashb ...


### Iteration Notes Template
- Version tested:
- What improved:
- What still failed:
- Next prompt change: